# 02c — EfficientNet-B4 Test Face Preparation

**Purpose:** Create native 380×380 RetinaFace test crops for the E3d EfficientNet-B4 experiment from the existing full-resolution FF++ and Celeb-DF pilot frames.

This notebook prepares **test data only**:

```text
dataset/test/{real,fake}
    → face_dataset_b4/test/{real,fake}

celebdf_v2/pilot_frames/{real,fake}
    → celebdf_v2/pilot_faces_b4/{real,fake}
```

It never reads from the existing 224×224 `face_dataset` or `pilot_faces`, never changes the saved manifests, and never trains or evaluates a detector.

## Experimental and preprocessing controls

- Crop directly from the saved full-resolution frames.
- Select the largest RetinaFace detection.
- Expand the detected face box using the established 15% margin.
- Resize the crop once to 380×380 and save ordinary RGB image files.
- Preserve source filenames so source-video identities remain recoverable.
- Safely resume complete outputs; never overwrite a valid crop during a normal rerun.
- Record missing detections and invalid crops rather than substituting images.
- Validate FF++ membership against `ffpp_video_split_manifest.json`.
- Validate Celeb-DF membership against `celebdf_pilot_manifest.json`.

## 1. Build, audit and rebuild controls

In [7]:
INSTALL_DEPENDENCIES = True

BUILD_FFPP_B4_TEST = True
BUILD_CELEBDF_B4_TEST = True

AUDIT_FFPP_B4_TEST = True
AUDIT_CELEBDF_B4_TEST = True

FORCE_REBUILD_FFPP_B4_TEST = False
FORCE_REBUILD_CELEBDF_B4_TEST = False
CONFIRM_FORCE_REBUILD = ""

FACE_SIZE = 380
FACE_MARGIN_RATIO = 0.15

assert FACE_SIZE == 380
assert FACE_MARGIN_RATIO == 0.15

if FORCE_REBUILD_FFPP_B4_TEST or FORCE_REBUILD_CELEBDF_B4_TEST:
    assert CONFIRM_FORCE_REBUILD == "REBUILD B4 TEST FACES", (
        "To force a rebuild, set CONFIRM_FORCE_REBUILD exactly to "
        "'REBUILD B4 TEST FACES'."
    )

print("Install dependencies:", INSTALL_DEPENDENCIES)
print("Build FF++ B4 test crops:", BUILD_FFPP_B4_TEST)
print("Build Celeb-DF B4 test crops:", BUILD_CELEBDF_B4_TEST)
print("Audit FF++ B4 test crops:", AUDIT_FFPP_B4_TEST)
print("Audit Celeb-DF B4 test crops:", AUDIT_CELEBDF_B4_TEST)
print("Face margin:", FACE_MARGIN_RATIO)
print("Saved face size:", f"{FACE_SIZE}x{FACE_SIZE}")

Install dependencies: True
Build FF++ B4 test crops: True
Build Celeb-DF B4 test crops: True
Audit FF++ B4 test crops: True
Audit Celeb-DF B4 test crops: True
Face margin: 0.15
Saved face size: 380x380


## 2. Mount Drive, install RetinaFace and resolve paths

In [8]:
from google.colab import drive
drive.mount("/content/drive")

import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "retina-face"],
        check=True,
    )

import cv2
import pandas as pd
from PIL import Image
from retinaface import RetinaFace

BASE_PATH = Path("/content/drive/MyDrive/deepfake_project")

FFPP_INPUT_ROOT = BASE_PATH / "dataset" / "test"
FFPP_OUTPUT_ROOT = BASE_PATH / "face_dataset_b4" / "test"
FFPP_MANIFEST_PATH = BASE_PATH / "ffpp_video_split_manifest.json"
FFPP_LOG_PATH = BASE_PATH / "results" / "ffpp_b4_test_face_build_log.csv"

CELEBDF_ROOT = BASE_PATH / "celebdf_v2"
CELEBDF_INPUT_ROOT = CELEBDF_ROOT / "pilot_frames"
CELEBDF_OUTPUT_ROOT = CELEBDF_ROOT / "pilot_faces_b4"
CELEBDF_MANIFEST_PATH = CELEBDF_ROOT / "celebdf_pilot_manifest.json"
CELEBDF_LOG_PATH = CELEBDF_ROOT / "celebdf_pilot_faces_b4_build_log.csv"

CONFIG_PATH = BASE_PATH / "results" / "b4_test_face_preparation_config.json"
VALID_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for required in (
    FFPP_INPUT_ROOT,
    FFPP_MANIFEST_PATH,
    CELEBDF_INPUT_ROOT,
    CELEBDF_MANIFEST_PATH,
):
    if not required.exists():
        raise FileNotFoundError(required)

FFPP_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
CELEBDF_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", BASE_PATH)
print("FF++ source:", FFPP_INPUT_ROOT)
print("FF++ B4 output:", FFPP_OUTPUT_ROOT)
print("Celeb-DF source:", CELEBDF_INPUT_ROOT)
print("Celeb-DF B4 output:", CELEBDF_OUTPUT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/deepfake_project
FF++ source: /content/drive/MyDrive/deepfake_project/dataset/test
FF++ B4 output: /content/drive/MyDrive/deepfake_project/face_dataset_b4/test
Celeb-DF source: /content/drive/MyDrive/deepfake_project/celebdf_v2/pilot_frames
Celeb-DF B4 output: /content/drive/MyDrive/deepfake_project/celebdf_v2/pilot_faces_b4


## 3. Optional targeted force rebuild

In [3]:
def remove_generated_target(output_root, log_path, enabled):
    if not enabled:
        return

    # Both paths are explicit generated B4 targets; source datasets are untouched.
    if output_root.exists():
        shutil.rmtree(output_root)
        print("Removed generated folder:", output_root)
    if log_path.exists():
        log_path.unlink()
        print("Removed generated log:", log_path)


remove_generated_target(
    FFPP_OUTPUT_ROOT,
    FFPP_LOG_PATH,
    FORCE_REBUILD_FFPP_B4_TEST,
)
remove_generated_target(
    CELEBDF_OUTPUT_ROOT,
    CELEBDF_LOG_PATH,
    FORCE_REBUILD_CELEBDF_B4_TEST,
)

for root in (FFPP_OUTPUT_ROOT, CELEBDF_OUTPUT_ROOT):
    for class_name in ("real", "fake"):
        (root / class_name).mkdir(parents=True, exist_ok=True)

print("B4 output directories are ready.")

Removed generated folder: /content/drive/MyDrive/deepfake_project/face_dataset_b4/test
Removed generated log: /content/drive/MyDrive/deepfake_project/results/ffpp_b4_test_face_build_log.csv
Removed generated folder: /content/drive/MyDrive/deepfake_project/celebdf_v2/pilot_faces_b4
Removed generated log: /content/drive/MyDrive/deepfake_project/celebdf_v2/celebdf_pilot_faces_b4_build_log.csv
B4 output directories are ready.


## 4. Verify source frames and saved manifests

In [9]:
def list_images(folder):
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in VALID_IMAGE_EXTENSIONS
    )


with FFPP_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    ffpp_manifest = json.load(handle)

with CELEBDF_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    celebdf_manifest = json.load(handle)

source_rows = []
for dataset_name, input_root in (
    ("FF++", FFPP_INPUT_ROOT),
    ("Celeb-DF", CELEBDF_INPUT_ROOT),
):
    for class_name in ("real", "fake"):
        paths = list_images(input_root / class_name)
        source_rows.append({
            "dataset": dataset_name,
            "class": class_name,
            "source_frames": len(paths),
            "folder_exists": (input_root / class_name).is_dir(),
        })

source_table = pd.DataFrame(source_rows)
display(source_table)

ffpp_counts = {
    class_name: len(list_images(FFPP_INPUT_ROOT / class_name))
    for class_name in ("real", "fake")
}
celebdf_counts = {
    class_name: len(list_images(CELEBDF_INPUT_ROOT / class_name))
    for class_name in ("real", "fake")
}

assert ffpp_counts == {"real": 300, "fake": 300}, ffpp_counts
assert celebdf_counts == {"real": 200, "fake": 200}, celebdf_counts
assert len(ffpp_manifest["classes"]["real"]["test"]) == 30
assert len(ffpp_manifest["classes"]["fake"]["test"]) == 30
assert len(celebdf_manifest["real_videos"]) == 20
assert len(celebdf_manifest["fake_videos"]) == 20
assert celebdf_manifest["frames_per_video"] == 10

print("PASS: source frames and manifests are ready.")

,dataset,class,source_frames,folder_exists
0,FF++,real,300,True
1,FF++,fake,300,True
2,Celeb-DF,real,200,True
3,Celeb-DF,fake,200,True


PASS: source frames and manifests are ready.


## 5. RetinaFace largest-face crop definition

In [10]:
def crop_largest_face(frame_path):
    image_bgr = cv2.imread(str(frame_path))
    if image_bgr is None:
        return None, "unreadable frame"

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    try:
        detections = RetinaFace.detect_faces(image_rgb)
    except Exception:
        return None, "RetinaFace error"

    if not isinstance(detections, dict) or not detections:
        return None, "no face detected"

    valid_faces = []
    for detection in detections.values():
        area = detection.get("facial_area")
        if area is None or len(area) != 4:
            continue
        x1, y1, x2, y2 = map(int, area)
        width = x2 - x1
        height = y2 - y1
        if width > 0 and height > 0:
            valid_faces.append((width * height, x1, y1, x2, y2))

    if not valid_faces:
        return None, "no valid face box"

    _, x1, y1, x2, y2 = max(valid_faces, key=lambda item: item[0])
    face_width = x2 - x1
    face_height = y2 - y1
    margin_x = int(face_width * FACE_MARGIN_RATIO)
    margin_y = int(face_height * FACE_MARGIN_RATIO)

    image_height, image_width = image_bgr.shape[:2]
    crop_x1 = max(0, x1 - margin_x)
    crop_y1 = max(0, y1 - margin_y)
    crop_x2 = min(image_width, x2 + margin_x)
    crop_y2 = min(image_height, y2 + margin_y)

    crop = image_bgr[crop_y1:crop_y2, crop_x1:crop_x2]
    if crop.size == 0:
        return None, "invalid crop"

    interpolation = (
        cv2.INTER_AREA
        if crop.shape[0] >= FACE_SIZE and crop.shape[1] >= FACE_SIZE
        else cv2.INTER_LINEAR
    )
    crop = cv2.resize(crop, (FACE_SIZE, FACE_SIZE), interpolation=interpolation)
    return crop, "saved"


print("Crop policy: largest RetinaFace detection + 15% margin")
print("Output resolution:", f"{FACE_SIZE}x{FACE_SIZE}")

Crop policy: largest RetinaFace detection + 15% margin
Output resolution: 380x380


## 6. Build or safely resume both B4 test datasets

In [11]:
def existing_output_is_valid(output_path):
    if not output_path.is_file():
        return False
    try:
        with Image.open(output_path) as image:
            return image.size == (FACE_SIZE, FACE_SIZE)
    except Exception:
        return False


def build_b4_faces(dataset_name, input_root, output_root, log_path, enabled):
    if not enabled:
        print(f"{dataset_name} B4 build skipped.")
        return []

    start_time = time.time()
    rows = []
    for class_name in ("real", "fake"):
        source_paths = list_images(input_root / class_name)
        print(f"Processing {dataset_name} {class_name}: {len(source_paths)} frames")

        for index, source_path in enumerate(source_paths, start=1):
            output_path = output_root / class_name / source_path.name

            if output_path.exists():
                if not existing_output_is_valid(output_path):
                    raise RuntimeError(
                        f"Existing output is not a valid {FACE_SIZE}x{FACE_SIZE} crop: "
                        f"{output_path}. Use the explicit force-rebuild controls."
                    )
                status = "already valid"
            else:
                crop, status = crop_largest_face(source_path)
                if crop is not None:
                    saved = cv2.imwrite(str(output_path), crop)
                    if not saved:
                        raise RuntimeError(f"Could not save crop: {output_path}")

            rows.append({
                "dataset": dataset_name,
                "class": class_name,
                "filename": source_path.name,
                "status": status,
                "output_path": str(output_path) if status in {"saved", "already valid"} else None,
            })

            if index % 100 == 0 or index == len(source_paths):
                print(f"  {index}/{len(source_paths)}")

    log_table = pd.DataFrame(rows)
    log_table.to_csv(log_path, index=False)
    elapsed_minutes = (time.time() - start_time) / 60
    print(f"{dataset_name} B4 preparation completed in {elapsed_minutes:.2f} minutes.")
    print("Build log:", log_path)
    display(log_table.groupby(["dataset", "class", "status"]).size().rename("frames"))
    return rows


ffpp_build_rows = build_b4_faces(
    "FF++",
    FFPP_INPUT_ROOT,
    FFPP_OUTPUT_ROOT,
    FFPP_LOG_PATH,
    BUILD_FFPP_B4_TEST,
)

celebdf_build_rows = build_b4_faces(
    "Celeb-DF",
    CELEBDF_INPUT_ROOT,
    CELEBDF_OUTPUT_ROOT,
    CELEBDF_LOG_PATH,
    BUILD_CELEBDF_B4_TEST,
)

Processing FF++ real: 300 frames
  100/300
  200/300
  300/300
Processing FF++ fake: 300 frames
  100/300
  200/300
  300/300
FF++ B4 preparation completed in 0.12 minutes.
Build log: /content/drive/MyDrive/deepfake_project/results/ffpp_b4_test_face_build_log.csv


dataset  class  status          
FF++     fake   already valid       299
                no face detected      1
         real   already valid       277
                no face detected      8
                saved                15
Name: frames, dtype: int64

Processing Celeb-DF real: 200 frames
  100/200
  200/200
Processing Celeb-DF fake: 200 frames
  100/200
  200/200
Celeb-DF B4 preparation completed in 0.01 minutes.
Build log: /content/drive/MyDrive/deepfake_project/celebdf_v2/celebdf_pilot_faces_b4_build_log.csv


dataset   class  status       
Celeb-DF  fake   already valid    200
          real   already valid    200
Name: frames, dtype: int64

## 7. Audit FF++ B4 test crops

In [12]:
def ffpp_video_id(path):
    if "_frame_" not in path.stem:
        raise ValueError(f"Cannot recover FF++ video id from: {path.name}")
    return path.stem.rsplit("_frame_", 1)[0]


def audit_dimensions(paths):
    dimensions = {}
    for path in paths:
        with Image.open(path) as image:
            dimensions[image.size] = dimensions.get(image.size, 0) + 1
    return dimensions


if AUDIT_FFPP_B4_TEST:
    expected_test_ids = {
        class_name: {
            Path(video_name).stem
            for video_name in ffpp_manifest["classes"][class_name]["test"]
        }
        for class_name in ("real", "fake")
    }

    rows = []
    for class_name in ("real", "fake"):
        source_paths = list_images(FFPP_INPUT_ROOT / class_name)
        output_paths = list_images(FFPP_OUTPUT_ROOT / class_name)
        source_names = {path.name for path in source_paths}
        output_names = {path.name for path in output_paths}
        output_video_ids = {ffpp_video_id(path) for path in output_paths}
        unexpected = output_names - source_names

        assert not unexpected, f"Unexpected FF++ B4 outputs: {sorted(unexpected)[:5]}"
        assert output_video_ids == expected_test_ids[class_name], (
            f"FF++ {class_name} B4 crops do not cover exactly the expected "
            f"test videos. Missing videos: "
            f"{sorted(expected_test_ids[class_name] - output_video_ids)[:5]}"
        )

        dimensions = audit_dimensions(output_paths)
        assert set(dimensions) <= {(FACE_SIZE, FACE_SIZE)}, dimensions

        rows.append({
            "class": class_name,
            "source_frames": len(source_paths),
            "b4_face_crops": len(output_paths),
            "missing_crops": len(source_names - output_names),
            "source_videos": len(output_video_ids),
            "expected_videos": len(expected_test_ids[class_name]),
            "dimensions": dimensions,
            "unexpected_files": len(unexpected),
        })

    ffpp_audit_table = pd.DataFrame(rows)
    display(ffpp_audit_table)
    print("PASS: FF++ B4 crops preserve the clean test-video split and are 380x380.")

,class,source_frames,b4_face_crops,missing_crops,source_videos,expected_videos,dimensions,unexpected_files
0,real,300,292,8,30,30,"{(380, 380): 292}",0
1,fake,300,299,1,30,30,"{(380, 380): 299}",0


PASS: FF++ B4 crops preserve the clean test-video split and are 380x380.


## 8. Audit Celeb-DF B4 pilot crops

In [13]:
def celebdf_video_id(path):
    if "__frame_" not in path.stem:
        raise ValueError(f"Cannot recover Celeb-DF video id from: {path.name}")
    return path.stem.split("__frame_", 1)[0]


if AUDIT_CELEBDF_B4_TEST:
    expected_video_ids = {
        "real": {record["video_id"] for record in celebdf_manifest["real_videos"]},
        "fake": {record["video_id"] for record in celebdf_manifest["fake_videos"]},
    }

    rows = []
    for class_name in ("real", "fake"):
        source_paths = list_images(CELEBDF_INPUT_ROOT / class_name)
        output_paths = list_images(CELEBDF_OUTPUT_ROOT / class_name)
        source_names = {path.name for path in source_paths}
        output_names = {path.name for path in output_paths}
        output_video_ids = {celebdf_video_id(path) for path in output_paths}
        unexpected = output_names - source_names

        assert not unexpected, f"Unexpected Celeb-DF B4 outputs: {sorted(unexpected)[:5]}"
        assert output_video_ids == expected_video_ids[class_name], (
            f"Celeb-DF {class_name} B4 crops do not cover exactly the fixed "
            f"pilot videos. Missing videos: "
            f"{sorted(expected_video_ids[class_name] - output_video_ids)[:5]}"
        )

        dimensions = audit_dimensions(output_paths)
        assert set(dimensions) <= {(FACE_SIZE, FACE_SIZE)}, dimensions

        per_video_counts = {}
        for path in output_paths:
            video_id = celebdf_video_id(path)
            per_video_counts[video_id] = per_video_counts.get(video_id, 0) + 1

        rows.append({
            "class": class_name,
            "source_frames": len(source_paths),
            "b4_face_crops": len(output_paths),
            "missing_crops": len(source_names - output_names),
            "source_videos": len(output_video_ids),
            "expected_videos": len(expected_video_ids[class_name]),
            "minimum_per_video": min(per_video_counts.values()) if per_video_counts else 0,
            "maximum_per_video": max(per_video_counts.values()) if per_video_counts else 0,
            "dimensions": dimensions,
            "unexpected_files": len(unexpected),
        })

    celebdf_audit_table = pd.DataFrame(rows)
    display(celebdf_audit_table)
    print("PASS: Celeb-DF B4 crops match the fixed pilot identities and are 380x380.")

,class,source_frames,b4_face_crops,missing_crops,source_videos,expected_videos,minimum_per_video,maximum_per_video,dimensions,unexpected_files
0,real,200,200,0,20,20,10,10,"{(380, 380): 200}",0
1,fake,200,200,0,20,20,10,10,"{(380, 380): 200}",0


PASS: Celeb-DF B4 crops match the fixed pilot identities and are 380x380.


## 9. Save reproducibility configuration

In [14]:
configuration = {
    "purpose": "native EfficientNet-B4 test face preparation",
    "face_detector": "RetinaFace",
    "face_selection": "largest detected face",
    "face_margin_ratio": FACE_MARGIN_RATIO,
    "saved_face_size": [FACE_SIZE, FACE_SIZE],
    "ffpp_source": str(FFPP_INPUT_ROOT),
    "ffpp_output": str(FFPP_OUTPUT_ROOT),
    "ffpp_manifest": str(FFPP_MANIFEST_PATH),
    "celebdf_source": str(CELEBDF_INPUT_ROOT),
    "celebdf_output": str(CELEBDF_OUTPUT_ROOT),
    "celebdf_manifest": str(CELEBDF_MANIFEST_PATH),
    "augmentation": False,
    "normalisation_when_saved": False,
    "existing_224_crops_used_as_source": False,
}

with CONFIG_PATH.open("w", encoding="utf-8") as handle:
    json.dump(configuration, handle, indent=2)

print("Configuration saved to:", CONFIG_PATH)

Configuration saved to: /content/drive/MyDrive/deepfake_project/results/b4_test_face_preparation_config.json


## 10. Downstream evaluation

After both audits pass:

- E3d FF++ evaluation must read `face_dataset_b4/test`.
- E3d Celeb-DF evaluation must read `celebdf_v2/pilot_faces_b4`.
- Both evaluators must resize/check 380×380 input, scale pixels to `[0,1]`, apply no ImageNet mean/std normalisation, use threshold 0.5 and perform no retraining or adaptation.

The existing 224×224 datasets remain unchanged for the B0 experiments.